# IEEE39 inverter macOS comparison macOS comparison

Fresh Apple Silicon results for the IEEE39 inverter macOS comparison native DPsim, FMI Model Exchange (ME), and FMI Co-Simulation (CS) cases. The network, load event, timesteps, initialization, and metric definitions match the Linux comparison. The saved traces and metrics are under `build-fmi/ieee39-inverter/macos-ieee39-10s/`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "dpsim-fmi").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from a DPsim checkout")
BUILD = ROOT / "build-fmi"
if not BUILD.exists():
    BUILD = ROOT / "build-macos-fmi-latest"
RESULTS = BUILD / "ieee39-inverter" / "macos-ieee39-10s"
metrics = json.loads((RESULTS / "metrics.json").read_text())
steps = [int(x) for x in metrics["configuration"]["steps_us"]]
print("macOS results:", RESULTS)
print(metrics["configuration"])


def saved_path(value):
    path = Path(value)
    if path.exists():
        return path
    marker = "/build-fmi/"
    if marker in str(value):
        return BUILD / str(value).split(marker, 1)[1]
    return path

## Error and timing summary

In [ ]:
rows = []
for step in steps:
    case = metrics["cases"][str(step)]
    for method in ("me", "cs"):
        run = case["runtimes"][method]
        full = case["comparisons"][method]["full_run"]
        rows.append(
            {
                "timestep [µs]": step,
                "method": method.upper(),
                "PCC voltage RMS error [%]": full["voltage_rms_error_percent"],
                "inverter current RMS error [%]": full["current_rms_error_percent"],
                "solver time [s]": run["solver_wall_s"],
                "process time [s]": run["process_wall_s"],
                "max Newton iterations": run.get("newton_metrics", {}).get(
                    "max_iterations"
                ),
                "max scaled residual": run.get("newton_metrics", {}).get(
                    "max_scaled_residual"
                ),
            }
        )
summary = pd.DataFrame(rows).sort_values(["timestep [µs]", "method"])
display(
    summary.style.format({c: "{:.6g}" for c in summary.select_dtypes("number").columns})
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
for method, color in (("ME", "tab:blue"), ("CS", "tab:orange")):
    part = summary[summary.method == method]
    axes[0].plot(
        part["timestep [µs]"],
        part["PCC voltage RMS error [%]"],
        "o-",
        color=color,
        label=method,
    )
    axes[1].plot(
        part["timestep [µs]"],
        part["inverter current RMS error [%]"],
        "o-",
        color=color,
        label=method,
    )
    axes[2].plot(
        part["timestep [µs]"], part["process time [s]"], "o-", color=color, label=method
    )
for ax, title, ylabel in zip(
    axes,
    ("PCC voltage error", "Inverter current error", "Process runtime"),
    (
        "RMS error [% of native DPsim RMS]",
        "RMS error [% of native DPsim RMS]",
        "seconds",
    ),
):
    ax.set_xscale("log")
    ax.set_xlabel("Fixed timestep [µs]")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
plt.show()

step = 100 if 100 in steps else steps[len(steps) // 2]
case = metrics["cases"][str(step)]
event = metrics["configuration"]["load_step_s"]
me = pd.read_csv(saved_path(case["comparisons"]["me"]["load_step_zoom_trace"]))
cs = pd.read_csv(saved_path(case["comparisons"]["cs"]["load_step_zoom_trace"]))
t = (me.time_s - event) * 1000
fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True, constrained_layout=True)
axes[0].plot(t, me.native_v_a_V / 1000, "k--", label="Native DPsim")
axes[0].plot(t, me.me_v_a_V / 1000, color="tab:blue", label="DPsim ME")
axes[0].plot(
    (cs.time_s - event) * 1000, cs.cs_v_a_V / 1000, color="tab:orange", label="DPsim CS"
)
axes[1].plot(t, me.native_i_a_A, "k--", label="Native DPsim")
axes[1].plot(t, me.me_i_a_A, color="tab:blue", label="DPsim ME")
axes[1].plot(
    (cs.time_s - event) * 1000, cs.cs_i_a_A, color="tab:orange", label="DPsim CS"
)
axes[0].set_ylabel("PCC phase-a voltage [kV]")
axes[1].set_ylabel("Inverter phase-a current [A]")
axes[1].set_xlabel("Time from load step [ms]")
for ax in axes:
    ax.axvline(0, color="0.3", ls=":")
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.suptitle(f"IEEE39 inverter macOS comparison: {step} µs load-step response")
plt.show()